# 📄 hermes-acp-sdk — figures for the paper

> **📸 For the paper figure:** Menu **Run ▸ Run All Cells**, then screenshot each
> output cell marked **📸**. Use the **light** theme (Settings ▸ Theme ▸ JupyterLab Light)
> and zoom the browser to **~175%** (`Cmd`/`Ctrl` `+`) so the fonts are crisp. Crop to
> roughly **9 cm** wide (one card) for a two-column paper.

In [ ]:
import logging
for _n in ("mcp", "httpx", "uvicorn", "uvicorn.error", "uvicorn.access", "asyncio"):
    logging.getLogger(_n).setLevel(logging.WARNING)   # keep the output clean

from collections import Counter
from hermes_acp_sdk import HermesClient, AgentText, AgentThought, Usage, Finished
print("ready — run the two cells below and screenshot their output")

## 📸 Figure 1 — driving the Hermes agent: one typed event stream (screenshot below)

In [ ]:
events = []
async with HermesClient() as hermes:                     # spawns `hermes acp`, does the handshake
    print(f"connected to: {hermes.agent_name} {hermes.agent_version}\n")
    async with hermes.session() as s:                    # selects the model for you (the classic trap)
        async for ev in s.prompt("Think it through, then answer in one line: what is a Python traceback?"):
            events.append(ev)

print("events received :", dict(Counter(type(e).__name__ for e in events)))
thoughts = "".join(e.text for e in events if isinstance(e, AgentThought))
answer   = "".join(e.text for e in events if isinstance(e, AgentText))
usage    = [e for e in events if isinstance(e, Usage)]
if thoughts:
    print("\n🧠 reasoning >>>", thoughts[:180].strip(), "…")
print("\n💬 answer    >>>", answer.strip())
print("📊 usage     >>>", usage[-1] if usage else "n/a")

## 📸 Figure 2 — an isolated profile via clone_provider, no API key handled (screenshot below)

In [ ]:
async with HermesClient(profile="paper-demo", clone_provider=True, auto_prefix=True) as hermes:
    async with hermes.session() as s:
        print("profile        : app-paper-demo   (its own memory, sessions, skills)")
        print("provider       : cloned from the host — the app never handled an API key")
        print("models offered :", len(s.available_models))
        out = []
        async for ev in s.prompt("Reply with exactly: ISOLATED PROFILE OK"):
            if isinstance(ev, AgentText): out.append(ev.text)
        print("agent reply    :", "".join(out).strip())

---
*Figure 1: the SDK turns the Agent Client Protocol into a typed event stream — the agent's
reasoning, its answer, and real token usage — from any Python app, no Jupyter required.
Figure 2: `clone_provider=True` gives the app its own isolated Hermes profile while inheriting
the host's provider, so the app drives a fully-configured agent without ever touching a key.*